# Sequential agent (core logic)

This notebook contains the core implementation of the sequential decision-theoretic SCADA agent.

It includes:
- likelihood specification
- posterior update for the sequential daily variant
- utility-based decision rule

This notebook assumes that the required input objects have already been constructed elsewhere, in particular:
- `merged_15`
- `ft`

Injection experiments, evaluation routines, and other analysis steps are kept separate from the core agent implementation.

In [ ]:
# Likelihood specification

import numpy as np

# Hypothesis space

HYPOTHESES = ["Normal", "SensorFault", "CommFault"]

# Evidence domains
E_KV_VALUES       = ["normal", "abnormal", "unknown"]
E_EVENT_VALUES    = ["none", "present"]
E_EVENT_UC_VALUES = ["none", "present"]

# Likelihood model: P(E_kV, E_event, E_event_under_comm | H)

likelihood = {
    "Normal": {
        ("normal","none","none"):        0.40,
        ("normal","present","none"):     0.04,
        ("normal","present","present"):  0.0001,

        ("abnormal","none","none"):      0.05,
        ("abnormal","present","none"):   0.01,
        ("abnormal","present","present"):0.0001,

        ("unknown","none","none"):       0.45,
        ("unknown","present","none"):    0.05,
        ("unknown","present","present"): 0.0001,
    },

    "SensorFault": {
        ("normal","none","none"):        0.06,
        ("normal","present","none"):     0.05,
        ("normal","present","present"):  0.0001,

        ("abnormal","none","none"):      0.55,
        ("abnormal","present","none"):   0.25,
        ("abnormal","present","present"):0.01,

        ("unknown","none","none"):       0.05,
        ("unknown","present","none"):    0.03,
        ("unknown","present","present"): 0.01,
    },

    "CommFault": {
        ("normal","none","none"):        0.05,
        ("normal","present","none"):     0.05,
        ("normal","present","present"):  0.10,

        ("abnormal","none","none"):      0.05,
        ("abnormal","present","none"):   0.05,
        ("abnormal","present","present"):0.15,

        ("unknown","none","none"):       0.35,
        ("unknown","present","none"):    0.10,
        ("unknown","present","present"): 0.10,
    }
}


# Normalize likelihood tables

for h, lh in likelihood.items():
    Z = float(sum(lh.values()))
    assert Z > 0, f"Zero likelihood mass for {h}"
    for k in list(lh.keys()):
        lh[k] = float(lh[k]) / Z

for h in HYPOTHESES:
    total = float(sum(likelihood[h].values()))
    assert abs(total - 1.0) < 1e-9, f"Likelihoods for {h} sum to {total}, not 1.0"

# Topology consistency likelihood: P(E_topo_consistency | H)

TOPO_LH = {
    "Normal": {
        "consistent":   0.90,
        "inconsistent": 0.01,
        "unknown":      0.09,
    },
    "SensorFault": {
        "consistent":   0.20,
        "inconsistent": 0.70,
        "unknown":      0.10,
    },
    "CommFault": {
        "consistent":   0.30,
        "inconsistent": 0.40,
        "unknown":      0.30,
    },
}

for h in HYPOTHESES:
    Z = float(sum(TOPO_LH[h].values()))
    assert abs(Z - 1.0) < 1e-9, f"TOPO_LH for {h} sums to {Z}"

# Event sequence likelihood: P(E_event_sequence | H)

SEQUENCE_LH = {
    "Normal":      {"none": 0.98, "present": 0.02},
    "SensorFault": {"none": 0.30, "present": 0.70},
    "CommFault":   {"none": 0.15, "present": 0.85},
}

for h in HYPOTHESES:
    Z = float(sum(SEQUENCE_LH[h].values()))
    assert abs(Z - 1.0) < 1e-9, f"SEQUENCE_LH for {h} sums to {Z}"



In [ ]:
# Posterior update (sequential, daily)


import pandas as pd
import numpy as np

# Contract checks

assert "merged_15" in globals(), "Mangler merged_15"
assert "likelihood" in globals(), "Mangler likelihood (16.14)"
assert "TOPO_LH" in globals(), "Mangler TOPO_LH (16.14)"
assert "SEQUENCE_LH" in globals(), "Mangler SEQUENCE_LH (16.14)"

REQUIRED_COLS = ["fault_loc_key", "window_start_15min", "E_kV_15min", "E_event_any", "E_comm_fail_any", "E_topo_consistency"]
for c in REQUIRED_COLS:
    assert c in merged_15.columns, f"merged_15 manglar {c}"

AGENT_PRIORS = {"Normal": 0.90, "SensorFault": 0.05, "CommFault": 0.05}
HYPOTHESES = ["Normal", "SensorFault", "CommFault"]

INTERLOCK_LH = {
    "Normal":      {"none": 1.0, "present": 0.001},
    "SensorFault": {"none": 1.0, "present": 0.70},
    "CommFault":   {"none": 1.0, "present": 0.75},
}
INTERLOCK_AGG_LH = {
    "Normal":      {"none": 1.0, "few": 0.05, "many": 0.001},
    "SensorFault": {"none": 1.0, "few": 0.60, "many": 0.15},
    "CommFault":   {"none": 1.0, "few": 0.80, "many": 0.95},
}

def _safe_log(a):
    a = np.asarray(a, dtype=float)
    a = np.clip(a, 1e-300, None)
    return np.log(a)

def _normalize_vec(v: np.ndarray) -> np.ndarray:
    s = float(v.sum())
    if s <= 0 or not np.isfinite(s):
        return np.ones_like(v) / len(v)
    return v / s

def posterior_update_sequential_daily(
    df: pd.DataFrame,
    likelihood_dict: dict,
    topo_lh: dict,
    seq_lh: dict,
    base_priors: dict = AGENT_PRIORS,
    mask=None,
) -> pd.DataFrame:
    out = df.copy()

    if mask is None:
        mask = pd.Series(True, index=out.index)
    else:
        mask = pd.Series(mask, index=out.index) if not isinstance(mask, pd.Series) else mask

    
    # Evidence transformation (compact representation)
   
    out.loc[mask, "E_kV"] = (
        out.loc[mask, "E_kV_15min"]
        .map({
            "present": "normal",
            "absent_or_invalid": "abnormal",
            "normal": "normal",
            "abnormal": "abnormal",
            "unknown": "unknown",
        })
        .fillna("unknown")
    )

    out.loc[mask, "E_event"] = (
        out.loc[mask, "E_event_any"]
        .fillna(0).astype(int)
        .map({1: "present", 0: "none"})
        .fillna("none")
    )

    out["E_comm_fail_any"] = out["E_comm_fail_any"].fillna(0).astype(int).clip(0, 1)

    out.loc[mask, "E_event_under_comm"] = np.where(
        (out.loc[mask, "E_event_any"].fillna(0).astype(int).eq(1))
        & (out.loc[mask, "E_comm_fail_any"].eq(1)),
        "present",
        "none",
    )

    if "E_event_sequence" not in out.columns:
        out["E_event_sequence"] = "none"
    out["E_event_sequence"] = out["E_event_sequence"].fillna("none")
    out.loc[~out["E_event_sequence"].isin(["none", "present"]), "E_event_sequence"] = "none"

    if "E_interlock" not in out.columns:
        out["E_interlock"] = "none"
    out["E_interlock"] = out["E_interlock"].fillna("none")
    out.loc[~out["E_interlock"].isin(["none", "present"]), "E_interlock"] = "none"

    if "E_interlock_agg" not in out.columns:
        out["E_interlock_agg"] = "none"
    out["E_interlock_agg"] = out["E_interlock_agg"].fillna("none")
    out.loc[~out["E_interlock_agg"].isin(["none", "few", "many"]), "E_interlock_agg"] = "none"


    # Compact view + day
  
    view = out.loc[mask, [
        "fault_loc_key", "window_start_15min",
        "E_kV", "E_event", "E_event_under_comm",
        "E_topo_consistency", "E_event_sequence",
        "E_interlock", "E_interlock_agg"
    ]].copy()

    view["fault_loc_key"] = view["fault_loc_key"].astype(str).str.strip()
    view["window_start_15min"] = pd.to_datetime(view["window_start_15min"], utc=True, errors="coerce")
    view = view.dropna(subset=["window_start_15min"]).copy()
    view["day"] = view["window_start_15min"].dt.floor("D")

   
    # Likelihood evaluation using predefined evidence keys
    
    all_keys = list(likelihood_dict["Normal"].keys()) 
    key_to_i = {k: i for i, k in enumerate(all_keys)}

    keys = list(zip(
        view["E_kV"].to_numpy(),
        view["E_event"].to_numpy(),
        view["E_event_under_comm"].to_numpy()
    ))
    key_idx = np.fromiter((key_to_i.get(k, -1) for k in keys), dtype=int, count=len(keys))

  
    if (key_idx < 0).any():
        bad_i = int(np.where(key_idx < 0)[0][0])
        raise KeyError(f"Key not in likelihood table: {keys[bad_i]} (row {bad_i} in view)")

    topo_code = view["E_topo_consistency"].map({"consistent": 0, "inconsistent": 1, "unknown": 2}).fillna(2).astype(int).to_numpy()
    seq_code  = view["E_event_sequence"].map({"none": 0, "present": 1}).fillna(0).astype(int).to_numpy()
    il_code   = view["E_interlock"].map({"none": 0, "present": 1}).fillna(0).astype(int).to_numpy()
    ia_code   = view["E_interlock_agg"].map({"none": 0, "few": 1, "many": 2}).fillna(0).astype(int).to_numpy()

    ll_df = pd.DataFrame({
        "fault_loc_key": view["fault_loc_key"].to_numpy(),
        "day": view["day"].to_numpy(),
    })

    for h in HYPOTHESES:
        lk = np.array([likelihood_dict[h][k] for k in all_keys], dtype=float)
        tp = np.array([topo_lh[h][t] for t in ["consistent", "inconsistent", "unknown"]], dtype=float)
        sq = np.array([seq_lh[h][t] for t in ["none", "present"]], dtype=float)
        il = np.array([INTERLOCK_LH[h][t] for t in ["none", "present"]], dtype=float)
        ia = np.array([INTERLOCK_AGG_LH[h][t] for t in ["none", "few", "many"]], dtype=float)

        ll_df[f"ll_{h}"] = (
            _safe_log(lk[key_idx]) +
            _safe_log(tp[topo_code]) +
            _safe_log(sq[seq_code]) +
            _safe_log(il[il_code]) +
            _safe_log(ia[ia_code])
        )

    
    # Aggregate likelihoods per (location, day) and update priors sequentially
    
    g = (
        ll_df.groupby(["fault_loc_key", "day"], as_index=False)[[f"ll_{h}" for h in HYPOTHESES]]
            .sum()
            .sort_values(["fault_loc_key", "day"])
    )

    base_prior_vec = np.array([base_priors[h] for h in HYPOTHESES], dtype=float)

    post_rows = []
    cur_loc = None
    prior_vec = None

    for r in g.itertuples(index=False):
        loc = r.fault_loc_key
        if loc != cur_loc:
            cur_loc = loc
            prior_vec = base_prior_vec.copy()

        ll_vec = np.array([getattr(r, f"ll_{h}") for h in HYPOTHESES], dtype=float)
        un = np.exp(ll_vec) * prior_vec
        post_vec = _normalize_vec(un)

        post_rows.append((loc, r.day, float(post_vec[0]), float(post_vec[1]), float(post_vec[2])))

        prior_vec = post_vec

    gpost = pd.DataFrame(post_rows, columns=["fault_loc_key", "day", "P_Normal_day", "P_SensorFault_day", "P_CommFault_day"])

    
    # Assign daily posterior to all observations within each (location, day)
    
    out.loc[mask, "fault_loc_key"] = out.loc[mask, "fault_loc_key"].astype(str).str.strip()
    out.loc[mask, "window_start_15min"] = pd.to_datetime(out.loc[mask, "window_start_15min"], utc=True, errors="coerce")
    out.loc[mask, "day"] = out.loc[mask, "window_start_15min"].dt.floor("D")

    tmp = out.loc[mask, ["fault_loc_key", "day"]].merge(
        gpost, on=["fault_loc_key", "day"], how="left", validate="m:1"
    )

    out.loc[mask, "P_Normal"] = tmp["P_Normal_day"].to_numpy()
    out.loc[mask, "P_SensorFault"] = tmp["P_SensorFault_day"].to_numpy()
    out.loc[mask, "P_CommFault"] = tmp["P_CommFault_day"].to_numpy()

    out = out.drop(columns=["day"], errors="ignore")
    return out




In [ ]:
# Utilities, expected utility, and decision rule

import pandas as pd
import numpy as np

# Preconditions

assert "merged_15" in globals(), "Missing merged_15 (run posterior update first)"
assert "ft" in globals(), "Missing function table (ft)"

REQUIRED_POST_COLS = ["P_Normal", "P_SensorFault", "P_CommFault"]
for c in REQUIRED_POST_COLS:
    assert c in merged_15.columns, f"Missing {c} (run posterior update first)"


# Build utility mapping per fault location

UTIL_COLS = [
    "SensorIntegrity_Utility",
    "StatusConsistency_Utility",
    "CyberSecurity_Utility",
    "DataIntegrity_Utility",
    "DigitalTwinImpact_Utility",
]

def build_utils_by_loc_from_ft(ft_df: pd.DataFrame) -> pd.DataFrame:
    missing = [c for c in (["fault_loc_key"] + UTIL_COLS) if c not in ft_df.columns]
    assert not missing, f"Function table missing utility columns: {missing}"

    tmp = ft_df[["fault_loc_key"] + UTIL_COLS].copy()
    tmp = tmp[tmp["fault_loc_key"].notna()].copy()
    tmp["fault_loc_key"] = tmp["fault_loc_key"].astype(str).str.strip()

    for c in UTIL_COLS:
        tmp[c] = pd.to_numeric(tmp[c], errors="coerce")

    # Worst-case aggregation per location

    utils_by_loc = (
        tmp.groupby("fault_loc_key")[UTIL_COLS]
           .min()
           .add_suffix("_worst")
           .reset_index()
    )

    return utils_by_loc


# Build once

if "UTILS_BY_LOC" not in globals():
    UTILS_BY_LOC = build_utils_by_loc_from_ft(ft)


# Decision update

def decision_update(df: pd.DataFrame, utils_by_loc: pd.DataFrame, mask=None) -> pd.DataFrame:
    out = df.copy()

    # Mask as positional boolean array

    if mask is None:
        mask_arr = np.ones(len(out), dtype=bool)
    else:
        mask_arr = np.asarray(mask, dtype=bool)
        assert mask_arr.shape[0] == len(out), "Mask length mismatch"

    # Ensure consistent fault location keys

    out["fault_loc_key"] = out["fault_loc_key"].astype("string").str.strip()

    # Ensure clean utility merge (avoid duplicate columns)

    worst_cols = [c for c in out.columns if c.endswith("_worst")] + ["n_rows"]
    out = out.drop(columns=worst_cols, errors="ignore")

    # Merge utilities

    out = out.merge(utils_by_loc, on="fault_loc_key", how="left", validate="m:1")

    NEED_UTIL = [
        "SensorIntegrity_Utility_worst",
        "StatusConsistency_Utility_worst",
        "CyberSecurity_Utility_worst",
        "DataIntegrity_Utility_worst",
        "DigitalTwinImpact_Utility_worst",
    ]

    for c in NEED_UTIL:
        assert c in out.columns, f"Missing {c} after merge"

    # Extract utility components

    U_SI = out.loc[mask_arr, "SensorIntegrity_Utility_worst"]
    U_SC = out.loc[mask_arr, "StatusConsistency_Utility_worst"]
    U_CS = out.loc[mask_arr, "CyberSecurity_Utility_worst"]
    U_DI = out.loc[mask_arr, "DataIntegrity_Utility_worst"]
    U_DT = out.loc[mask_arr, "DigitalTwinImpact_Utility_worst"]

    # Action costs

    COST = {"Ignore": 0.00, "Inspect": -0.10, "Isolate": -0.15}

    def mean2(a, b):
        return np.nanmean(np.vstack([a.to_numpy(), b.to_numpy()]), axis=0)


    # Utility function U(action, hypothesis)

    out.loc[mask_arr, "U_Ignore_Normal"]      = COST["Ignore"]
    out.loc[mask_arr, "U_Ignore_SensorFault"] = COST["Ignore"] + (-mean2(U_SI, U_DI))
    out.loc[mask_arr, "U_Ignore_CommFault"]   = COST["Ignore"] + (-mean2(U_CS, U_DI))

    out.loc[mask_arr, "U_Inspect_Normal"]      = COST["Inspect"] - 0.02
    out.loc[mask_arr, "U_Inspect_SensorFault"] = COST["Inspect"] + mean2(U_SI, U_DI)
    out.loc[mask_arr, "U_Inspect_CommFault"]   = COST["Inspect"] + (U_DI.to_numpy() * 0.2)

    out.loc[mask_arr, "U_Isolate_Normal"]      = COST["Isolate"] - 0.10
    out.loc[mask_arr, "U_Isolate_SensorFault"] = COST["Isolate"] + 0.4 * mean2(U_SC, U_DI)
    out.loc[mask_arr, "U_Isolate_CommFault"]   = COST["Isolate"] + 1.2 * mean2(U_CS, U_DT)


    # Expected utility and decision rule

    out.loc[mask_arr, "EU_Ignore"] = (
        out.loc[mask_arr, "P_Normal"]      * out.loc[mask_arr, "U_Ignore_Normal"] +
        out.loc[mask_arr, "P_SensorFault"] * out.loc[mask_arr, "U_Ignore_SensorFault"] +
        out.loc[mask_arr, "P_CommFault"]   * out.loc[mask_arr, "U_Ignore_CommFault"]
    )

    out.loc[mask_arr, "EU_Inspect"] = (
        out.loc[mask_arr, "P_Normal"]      * out.loc[mask_arr, "U_Inspect_Normal"] +
        out.loc[mask_arr, "P_SensorFault"] * out.loc[mask_arr, "U_Inspect_SensorFault"] +
        out.loc[mask_arr, "P_CommFault"]   * out.loc[mask_arr, "U_Inspect_CommFault"]
    )

    out.loc[mask_arr, "EU_Isolate"] = (
        out.loc[mask_arr, "P_Normal"]      * out.loc[mask_arr, "U_Isolate_Normal"] +
        out.loc[mask_arr, "P_SensorFault"] * out.loc[mask_arr, "U_Isolate_SensorFault"] +
        out.loc[mask_arr, "P_CommFault"]   * out.loc[mask_arr, "U_Isolate_CommFault"]
    )

    out.loc[mask_arr, "Decision"] = (
        out.loc[mask_arr, ["EU_Ignore", "EU_Inspect", "EU_Isolate"]]
        .idxmax(axis=1)
        .str.replace("EU_", "", regex=False)
    )

    assert set(out.loc[mask_arr, "Decision"].unique()) <= {"Ignore", "Inspect", "Isolate"}

    return out